### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="parkinsons_biomedical_voice_measurements",
    dataset_year="2007",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C59C74",
    download_description="""
We get the 2007 data from the UCI repository.

wget https://archive.ics.uci.edu/static/public/174/parkinsons.zip && unzip parkinsons.zip parkinsons.data && rm parkinsons.zip && mkdir -p local-data-warehouse/parkinsons_biomedical_voice_measurements && mv parkinsons.data local-data-warehouse/parkinsons_biomedical_voice_measurements/
""",
    # References
    academic_reference_bibtex="""@article{little2007exploiting,
  title={Exploiting nonlinear recurrence and fractal scaling properties for voice disorder detection},
  author={Little, Max and Mcsharry, Patrick and Roberts, Stephen and Costello, Declan and Moroz, Irene},
  journal={Nature Precedings},
  pages={1--1},
  year={2007},
  publisher={Nature Publishing Group UK London}
}
""",
    academic_reference_bibtex_key="little2007exploiting",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped", "WrongDomain"], # Maybe also temporal but not of importance for the task.
    curation_comments="""
We start with the data from UCI.

- We decode the group ID and session ID (time index per patient) from the name column.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="status",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="status",
    group_on="patient_id",
    group_time_on="session_number",
    group_labels="per_group",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "parkinsons.data")

# Split session number from patient ID, split on the last _
df[["patient_id", "session_number"]] = df["name"].str.rsplit("_", n=1, expand=True)
df["session_number"] = df["session_number"].astype(int)
df = df.drop(columns=["name"])

as_cat_type = ["status", "patient_id"]
df[as_cat_type] = df[as_cat_type].astype("category")
print("Loaded data shape:", df.shape)

df = df.sample(frac=1, random_state=42).sort_values(by=["patient_id", "session_number"]).reset_index(drop=True)

Loaded data shape: (195, 25)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 195
Columns: 25
Use sampling: False (sample size: 195)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['MDVP:Fo(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Fhi(Hz)', 'HNR', 'RPDE', 'D2', 'PPE', 'DFA', 'spread1', 'spread2']
Rows remaining as candidates after top-10 filter: 0 (of 195)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,MDVP:Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,MDVP:APQ,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE,patient_id,session_number
0,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,0.426,0.02182,0.03130,0.02971,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654,phon_R01_S01,1
1,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,0.626,0.03134,0.04518,0.04368,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674,phon_R01_S01,2
2,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,0.482,0.02757,0.03858,0.03590,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634,phon_R01_S01,3
3,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,0.517,0.02924,0.04005,0.03772,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975,phon_R01_S01,4
4,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,0.584,0.03490,0.04825,0.04465,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335,phon_R01_S01,5


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,status,category,0.0,0.0,2.0,"1, 0"
1,patient_id,category,0.0,0.0,32.0,"phon_R01_S21, phon_R01_S27, phon_R01_S35, phon_R01_S01, phon_R01_S06, phon_R01_S07, phon_R01_S04, phon_R01_S02, phon_R01_S10, phon_R01_S13"
2,MDVP:Fo(Hz),float64,0.0,0.0,195.0,"119.992, 122.4, 116.682, 116.676, 116.014, 120.552, 120.267, 107.332, 95.73, 95.056"
3,MDVP:Fhi(Hz),float64,0.0,0.0,195.0,"157.302, 148.65, 131.111, 137.871, 141.781, 131.162, 137.244, 113.84, 132.068, 120.103"
4,MDVP:Flo(Hz),float64,0.0,0.0,195.0,"74.997, 113.819, 111.555, 111.366, 110.655, 113.787, 114.82, 104.315, 91.754, 91.226"
5,MDVP:Jitter(%),float64,0.0,0.0,173.0,"0.0037, 0.0074, 0.0069, 0.0097, 0.0054, 0.0026, 0.003, 0.005, 0.005, 0.0048"
6,MDVP:Jitter(Abs),float64,0.0,0.0,19.0,"0.0, 0.0, 0.0, 0.0, 0.0001, 0.0001, 0.0001, 0.0001, 0.0001, 0.0"
7,MDVP:RAP,float64,0.0,0.0,155.0,"0.0017, 0.0016, 0.0025, 0.0043, 0.0016, 0.0013, 0.0032, 0.0027, 0.0037, 0.0022"
8,MDVP:PPQ,float64,0.0,0.0,165.0,"0.0033, 0.0018, 0.002, 0.0028, 0.0015, 0.0014, 0.0022, 0.0039, 0.0026, 0.0034"
9,Jitter:DDP,float64,0.0,0.0,180.0,"0.0051, 0.0111, 0.0129, 0.005, 0.007, 0.0062, 0.004, 0.005, 0.0075, 0.0035"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
MDVP:Fo(Hz),195.0,154.228641,41.390065,88.333000,260.105000
MDVP:Fhi(Hz),195.0,197.104918,91.491548,102.145000,592.030000
MDVP:Flo(Hz),195.0,116.324631,43.521413,65.476000,239.170000
MDVP:Jitter(%),195.0,0.006220,0.004848,0.001680,0.033160
MDVP:Jitter(Abs),195.0,0.000044,0.000035,0.000007,0.000260
MDVP:RAP,195.0,0.003306,0.002968,0.000680,0.021440
MDVP:PPQ,195.0,0.003446,0.002759,0.000920,0.019580
Jitter:DDP,195.0,0.009920,0.008903,0.002040,0.064330
MDVP:Shimmer,195.0,0.029709,0.018857,0.009540,0.119080
MDVP:Shimmer(dB),195.0,0.282251,0.194877,0.085000,1.302000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                            
patient_id 1     phon_R01_S21      7   3.59
           2     phon_R01_S27      7   3.59
           3     phon_R01_S35      7   3.59
           4     phon_R01_S01      6   3.08
           5     phon_R01_S06      6   3.08
status     1                1    147  75.38
           2                0     48  24.62

In [8]:
# Target Distribution
target_df

,count,pct
status,,
1,147,75.38
0,48,24.62


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Providing recommendations based on number of groups (32).
Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create a default grouped split with label-per-group to judge if model can learn to predict the state of a patient given a set of recordings.",
    splits=splits,
)

Using Stratified Grouped splits.
Using label-per-group grouped splits.
Creating index-based splits for 32 groups


Using Stratified IID splits.
Repeat 0, Fold 0:
            Train N: 127, Test N: 68
            Target Distribution:
            	Train target distribution: {1: 0.7637795275590551, 0: 0.23622047244094488}
            	Test target distribution: {1: 0.7352941176470589, 0: 0.2647058823529412}
            Group Distribution patient_id:
            	Train: 21
            	Test: 11
            
Repeat 0, Fold 1:
            Train N: 128, Test N: 67
            Target Distribution:
            	Train target distribution: {1: 0.765625, 0: 0.234375}
            	Test target distribution: {1: 0.7313432835820896, 0: 0.26865671641791045}
            Group Distribution patient_id:
            	Train: 21
            	Test: 11
            
Repeat 0, Fold 2:
            Train N: 135, Test N: 60
            Target Distribution:
            	Train target distribution: {1: 0.7333333333333333, 0: 0.26666666666666666}
            	Test target distribution: {1: 0.8, 0: 0.2}
            Group Distribution pa

Repeat 7, Fold 2:
            Train N: 135, Test N: 60
            Target Distribution:
            	Train target distribution: {1: 0.7333333333333333, 0: 0.26666666666666666}
            	Test target distribution: {1: 0.8, 0: 0.2}
            Group Distribution patient_id:
            	Train: 22
            	Test: 10
            
Repeat 8, Fold 0:
            Train N: 128, Test N: 67
            Target Distribution:
            	Train target distribution: {1: 0.765625, 0: 0.234375}
            	Test target distribution: {1: 0.7313432835820896, 0: 0.26865671641791045}
            Group Distribution patient_id:
            	Train: 21
            	Test: 11
            
Repeat 8, Fold 1:
            Train N: 128, Test N: 67
            Target Distribution:
            	Train target distribution: {1: 0.765625, 0: 0.234375}
            	Test target distribution: {1: 0.7313432835820896, 0: 0.26865671641791045}
            Group Distribution patient_id:
            	Train: 21
            	Tes

Repeat 13, Fold 2:
            Train N: 132, Test N: 63
            Target Distribution:
            	Train target distribution: {1: 0.7272727272727273, 0: 0.2727272727272727}
            	Test target distribution: {1: 0.8095238095238095, 0: 0.19047619047619047}
            Group Distribution patient_id:
            	Train: 22
            	Test: 10
            
Repeat 14, Fold 0:
            Train N: 128, Test N: 67
            Target Distribution:
            	Train target distribution: {1: 0.765625, 0: 0.234375}
            	Test target distribution: {1: 0.7313432835820896, 0: 0.26865671641791045}
            Group Distribution patient_id:
            	Train: 21
            	Test: 11
            
Repeat 14, Fold 1:
            Train N: 128, Test N: 67
            Target Distribution:
            	Train target distribution: {1: 0.765625, 0: 0.234375}
            	Test target distribution: {1: 0.7313432835820896, 0: 0.26865671641791045}
            Group Distribution patient_id:
      

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to parkinsons_biomedical_voice_measurements/019d738d-db39-7b5b-83dc-664e66da1854


019d738d-db39-7b5b-83dc-664e66da1854
c4908b24e28983026c388d101a00a06845d48aefa666d3dcd393be90657e875b
